# Homework 4

## Problem 3

### (a) Correlation between Climate Indices

**Calculate correlations among Nino 3.4, PDO, AMM, AMO, Hurricane Number, plus another two indices of your choice.**

Update: Some indices at NOAA/PSL website are out of maintenance. You can access hurricane number here: https://tropical.atmos.colostate.edu/Realtime/index.php?arch&loc=northatlantic 

In [ ]:
import pandas as pd 
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

Next we compute the correlation matrix

In [ ]:
hurricane = pd.read_csv("data/hurricane_number.csv", header=0, index_col=0,  sep='\\s+', engine='python')
hurricane_num = hurricane["Hurricanes"]
amm = (pd.read_csv("data/amm.csv", header=0, index_col=0,  sep='\\s+', engine='python')).mean(axis=1)
amo = (pd.read_csv("data/amo.csv", header=0, index_col=0,  sep='\\s+', engine='python')).mean(axis=1)
pdo = (pd.read_csv("data/pdo.csv", header=0, index_col=0,  sep='\\s+', engine='python')).mean(axis=1)
ao =  (pd.read_csv("data/ao.csv", header=0, index_col=0,  sep='\\s+', engine='python')).mean(axis=1)
enso_precip = (pd.read_csv("data/enso_precp.csv", header=0, index_col=0,  sep='\\s+', engine='python')).mean(axis=1)
nino = (pd.read_csv("data/nino3.4.csv", header=0, index_col=0,  sep='\\s+', engine='python')).mean(axis=1)
correlation = pd.DataFrame({"amm" : amm, "amo" : amo, "pdo" : pdo, "hurricane_num" : hurricane_num, "ao" : ao, "enso precipitatoin": enso_precip, "nino 3.4": nino}).dropna().corr()
print(correlation)

We can visualize the result better using a heat map

In [ ]:
plt.figure(figsize=(8, 6)) # Adjust figure size as needed
sns.heatmap(correlation, annot=True, cmap='coolwarm', vmin=-1, vmax=1)
plt.title("Pearson Correlation Matrix Between Climate Indices", fontsize=16, loc='center')

### (b) Climate Composite 

**Work with one index and rank its values. Identify the years with most positive and most negative values. Plot climate composites using precipitation data**


:::{tip}
I'll choose **amm** as my index.
:::

In [ ]:
fig, ax = plt.subplots() 
ax.plot(amm.index[0:len(amm)-1], amm.to_numpy()[0:len(amm)-1])
amm.sort_values()


Let's download the precipitation data that correspond to the lowest/highest *amm index* years.

In [13]:
import cdsapi

dataset = "reanalysis-era5-single-levels-monthly-means"
request = {
    "product_type": ["monthly_averaged_reanalysis"],
    "variable": ["total_precipitation"],
    "year": [
        "1998", "2010", "2016", '2017', '1952',
        "1974", "1976", "1972", '1971', '1975'
    ],
    "month": [
        "01", "02", "03",
        "04", "05", "06",
        "07", "08", "09",
        "10", "11", "12"
    ],
    "time": ["00:00"],
    "data_format": "grib",
    "download_format": "unarchived"
}

client = cdsapi.Client()
client.retrieve(dataset, request).download()


2026-03-13 12:55:14,259 INFO Request ID is 9b16c3e8-ec5d-492c-be01-d5f4f071ff3c
2026-03-13 12:55:14,428 INFO status has been updated to accepted
2026-03-13 12:55:23,846 INFO status has been updated to running
2026-03-13 12:55:48,366 INFO status has been updated to successful


'f78a7d9753063a0963de410330a43a31.grib'

Again using my library **climavis**, we can do some visualization:

In [1]:
import climavis as cv
import xarray
array = xarray.open_dataset("data/precipitation.grib")
print(array)

# tp1 = array['tp'][0:12].mean(dim='time')
# tp2 = array['tp'][12:24].mean(dim='time')
# tp = xarray.concat([tp1,tp2], dim='time').to_dataset(name="tp")
# print(tp['tp'])



<xarray.Dataset> Size: 498MB
Dimensions:     (time: 120, latitude: 721, longitude: 1440)
Coordinates:
  * time        (time) datetime64[ns] 960B 1951-12-31T18:00:00 ... 2017-11-30...
    valid_time  (time) datetime64[ns] 960B ...
  * latitude    (latitude) float64 6kB 90.0 89.75 89.5 ... -89.5 -89.75 -90.0
  * longitude   (longitude) float64 12kB 0.0 0.25 0.5 0.75 ... 359.2 359.5 359.8
    number      int64 8B ...
    step        timedelta64[ns] 8B ...
    surface     float64 8B ...
Data variables:
    tp          (time, latitude, longitude) float32 498MB ...
Attributes:
    GRIB_edition:            1
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-03-13T13:01 GRIB to CDM+CF via cfgrib-0.9.1...


In [ ]:
data_path = {
    "tp" : "data/precipitation.grib" 
}

ds_tp = cv.DataSet("total precipitation", data_paths = data_path, time_steps=72, unit='m')

intervals = [(12 * i, 12 * (i + 1)) for i in range(10)]

ds_tp.averageOverTime(intervals, new_time_unit='years', new_time_steps=10, new_time_sequence=[
        "1998", "2010", "2016", '2017', '1952',
        "1974", "1976", "1972", '1971', '1975'
    ])

print(ds_tp.S)

# intervals_ = [(0, 5), (5, 10)]
# ds_tp.averageOverTime(intervals, new_time_steps=2, new_time_sequence=['+', '-'])

vis_param = {
    "task_name" : "yearly_total_precipitation",
    "data_set" : ds_tp, 
    "outputs_dir" : "output"
}

print(ds_tp.mode)

vis = cv.Visualize(**vis_param)
vis.populate_frame(title="AMM Total Precipitation Composite")

<xarray.Dataset> Size: 42MB
Dimensions:    (latitude: 721, longitude: 1440, time: 10)
Coordinates:
  * latitude   (latitude) float64 6kB 90.0 89.75 89.5 ... -89.5 -89.75 -90.0
  * longitude  (longitude) float64 12kB 0.0 0.25 0.5 0.75 ... 359.2 359.5 359.8
    number     int64 8B 0
    step       timedelta64[ns] 8B 12:00:00
    surface    float64 8B 0.0
Dimensions without coordinates: time
Data variables:
    tp         (time, latitude, longitude) float32 42MB 0.000579 ... 0.0001927
scalar
[Visualize-LOG] Directory 'output/yearly_total_precipitation' already exists
[Visualize-LOG] Directory 'output/yearly_total_precipitation/frame' already exists
[Visualize-LOG] Directory 'output/yearly_total_precipitation/animation' already exists
[Visualize LOG] Plotting frame in scalar mode


 60%|██████    | 6/10 [00:14<00:09,  2.31s/it]